Install needed dependencies libraries

In [1]:
!pip install transformers peft pyarrow datasets pandas accelerate bitsandbytes trl torch -q

zsh:1: command not found: pip


In [2]:
!pip install torchao --upgrade -q


zsh:1: command not found: pip


generate synthetic dataset for training

In [3]:
!uv add transformers peft pyarrow datasets pandas accelerate bitsandbytes trl torch -q

In [4]:
!uv add torchao --upgrade -q

In [5]:
import json
import random
from dataclasses import dataclass,asdict
from typing import List

random.seed(42)

TEMPLATES = {
    "запрос_информации": {
        "incoming": [
            "Уважаемые коллеги,\nПрошу предоставить информацию о {topic} в срок до {date}.\nС уважением,\n{sender}",
            "Добрый день!\nНам необходимы сведения относительно {topic}. Просим направить ответ до {date}.\nС уважением,\n{sender}",
            "Здравствуйте,\nОбращаемся с просьбой сообщить о текущем статусе {topic}. Ответ ожидаем до {date}.\n{sender}",
        ],
        "responses": {
            "вежливое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Благодарим за обращение. В ответ на Ваш запрос относительно {topic} сообщаем следующее:\n\n"
                "{response_body}\n\n"
                "При возникновении дополнительных вопросов просим обращаться к нам в рабочее время.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
            "краткое": (
                "Уважаемый(-ая) {sender},\n\n"
                "По вопросу {topic}: {response_body}\n\n"
                "С уважением, {responder}"
            ),
            "настойчивое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Подтверждаем получение Вашего запроса. Обращаем внимание, что {topic} требует "
                "детального рассмотрения. {response_body}\n\n"
                "Настоятельно просим учесть данную информацию при планировании дальнейших действий.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
        },
    },
    "жалоба": {
        "incoming": [
            "Уважаемое руководство,\nВынуждены сообщить о нарушении условий договора в части {topic}. "
            "Просим принять меры в срок до {date}.\nС уважением,\n{sender}",
            "Добрый день,\nВыражаем обеспокоенность в связи с {topic}. Ситуация требует немедленного урегулирования.\n{sender}",
        ],
        "responses": {
            "вежливое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Приносим свои извинения за возникшие неудобства в связи с {topic}.\n\n"
                "{response_body}\n\n"
                "Мы предпримем все необходимые меры для урегулирования данной ситуации "
                "и не допустим повторения подобных инцидентов в будущем.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
            "настойчивое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Ваша жалоба относительно {topic} принята к рассмотрению. "
                "{response_body}\n\n"
                "Обращаем внимание: повторное обращение по данному вопросу потребует "
                "привлечения профильных специалистов и займёт дополнительное время.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
            "краткое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Жалоба по {topic} принята. {response_body} Срок устранения — до {date}.\n\n"
                "С уважением, {responder}"
            ),
        },
    },
    "коммерческое_предложение": {
        "incoming": [
            "Уважаемые партнёры,\nПредлагаем рассмотреть наше коммерческое предложение по {topic}. "
            "Готовы обсудить детали на встрече {date}.\nС уважением,\n{sender}",
            "Добрый день,\nНаправляем предложение о сотрудничестве в сфере {topic}. "
            "Ждём Вашего ответа до {date}.\n{sender}",
        ],
        "responses": {
            "вежливое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Благодарим за проявленный интерес к сотрудничеству. Ваше предложение относительно "
                "{topic} внимательно рассмотрено.\n\n"
                "{response_body}\n\n"
                "Будем рады продолжить диалог и готовы организовать встречу в удобное для Вас время.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
            "краткое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Предложение по {topic} рассмотрено. {response_body}\n\n"
                "Готовы к переговорам. С уважением, {responder}"
            ),
            "настойчивое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Ваше предложение по {topic} изучено. {response_body}\n\n"
                "Просим направить окончательные условия до {date} для принятия решения.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
        },
    },
    "уведомление": {
        "incoming": [
            "Уважаемые коллеги,\nУведомляем Вас об изменениях в {topic}, вступающих в силу с {date}.\n{sender}",
            "Добрый день,\nНастоящим сообщаем, что {topic} будет изменён с {date}. Просим принять к сведению.\n{sender}",
        ],
        "responses": {
            "вежливое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Подтверждаем получение уведомления об изменениях в {topic}.\n\n"
                "{response_body}\n\n"
                "Благодарим за своевременное информирование.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
            "краткое": (
                "Уважаемый(-ая) {sender},\n\nИзменения по {topic} приняты к сведению. "
                "{response_body}\n\nС уважением, {responder}"
            ),
            "настойчивое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Уведомление об изменении {topic} получено. {response_body}\n\n"
                "Просим дополнительно направить официальный документ, подтверждающий данные изменения.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
        },
    },
    "согласование": {
        "incoming": [
            "Уважаемые коллеги,\nПросим согласовать {topic} до {date}.\n{sender}",
            "Добрый день,\nНаправляем на согласование {topic}. Ждём подтверждения.\n{sender}",
        ],
        "responses": {
            "вежливое": (
                "Уважаемый(-ая) {sender},\n\n"
                "Документы по {topic} получены и переданы на рассмотрение.\n\n"
                "{response_body}\n\n"
                "Решение будет направлено в установленные сроки.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
            "краткое": (
                "Уважаемый(-ая) {sender},\n\n{topic} — согласовано. {response_body}\n\nС уважением, {responder}"
            ),
            "настойчивое": (
                "Уважаемый(-ая) {sender},\n\n"
                "{topic} рассмотрено. {response_body}\n\n"
                "Обращаем внимание: в случае несоответствия требованиям документ будет возвращён на доработку.\n\n"
                "С уважением,\n{responder}\n{org}"
            ),
        },
    },
}


In [6]:
TOPICS = [
    "предоставления финансовой отчётности за Q3 2024",
    "обновления технической документации",
    "изменения условий договора поставки №123-2024",
    "реструктуризации задолженности",
    "переноса сроков выполнения работ",
    "внедрения новой системы документооборота",
    "проведения аудиторской проверки",
    "изменения тарифов на услуги",
    "согласования бюджета на 2025 год",
    "организации корпоративного обучения",
    "обновления регламента работы с клиентами",
    "привлечения нового поставщика услуг",
    "расторжения договора аренды",
    "оплаты счёта №789-2024",
    "устранения технической неисправности оборудования",
]

In [7]:
DATES = [
    "01.02.2025", "15.02.2025", "28.02.2025",
    "10.03.2025", "25.03.2025", "05.04.2025",
    "20.04.2025", "30.04.2025", "15.05.2025",
]

In [8]:
SENDERS = [
    "А.В. Петрова, ООО «ТехноСервис»",
    "И.И. Иванов, АО «РосИнвест»",
    "М.С. Сидорова, ЗАО «АльфаГрупп»",
    "В.П. Козлов, ИП Козлов В.П.",
    "Е.А. Смирнова, ПАО «МегаТрейд»",
    "Д.Р. Фёдоров, ООО «ПромСтрой»",
    "О.Н. Новикова, АНО «ЦифраПлюс»",
    "К.В. Морозов, ГК «ФинансГрупп»",
]

In [9]:
RESPONDERS = [
    "Н.К. Белова", "С.Г. Орлов", "Т.В. Романова",
    "А.Д. Крылов", "Л.И. Захарова", "П.С. Громов",
    "Е.М. Волкова", "Ф.О. Лебедев",
]

In [10]:
ORGS = [
    "ООО «СтандартГрупп»",
    "АО «ПартнёрИнвест»",
    "ЗАО «БизнесЛайн»",
    "ПАО «КонсультПро»",
    "ООО «ТехАльянс»",
]

In [11]:
RESPONSE_BODIES = {
    "запрос_информации": [
        "Запрошенные данные по указанному вопросу будут предоставлены в течение 5 рабочих дней. "
        "Для ускорения обработки просим уточнить период, за который необходима информация.",
        "Информация по данному вопросу находится в процессе подготовки. "
        "Ответственный специалист свяжется с Вами дополнительно.",
        "Сведения направлены в приложении к настоящему письму. "
        "Просим подтвердить получение.",
        "Запрашиваемая информация будет предоставлена после внутреннего согласования с профильным отделом.",
    ],
    "жалоба": [
        "Ситуация передана на рассмотрение профильному руководителю. "
        "Решение будет принято в течение 3 рабочих дней.",
        "По факту обращения проводится внутренняя проверка. О результатах сообщим дополнительно.",
        "Нарушение зафиксировано. Компенсация и устранение причин — в приоритете текущей недели.",
        "Виновная сторона установлена. Принимаются меры по недопущению повторных инцидентов.",
    ],
    "коммерческое_предложение": [
        "Предложенные условия требуют дополнительного анализа с участием финансового блока.",
        "Предложение представляет интерес. Готовы рассмотреть возможность пилотного сотрудничества.",
        "Условия частично приемлемы. Предлагаем обсудить корректировку ценовых параметров.",
        "Предложение соответствует нашим текущим потребностям. Направляем встречные условия.",
    ],
    "уведомление": [
        "Данные изменения учтены во внутренних регламентах.",
        "Соответствующие подразделения уведомлены о вступающих в силу изменениях.",
        "Просим направить официальный приказ для отражения в договорной базе.",
        "Изменения приняты к исполнению с указанной даты.",
    ],
    "согласование": [
        "Замечания по тексту документа направлены отдельным письмом.",
        "Документ согласован без замечаний. Подписанный экземпляр направляем в ответном вложении.",
        "Согласование возможно после устранения замечаний, перечисленных в приложении.",
        "Вопрос передан на дополнительное рассмотрение юридическому департаменту.",
    ],
}

INSTRUCTIONS = [
    "Составьте вежливый профессиональный ответ на входящее деловое письмо.",
    "Напишите краткий деловой ответ на следующее письмо.",
    "Подготовьте настойчивый, но вежливый ответ на деловое письмо.",
    "Сформируйте официальный ответ на входящее письмо в деловом стиле.",
    "Составьте ответное письмо в соответствии с деловым этикетом.",
]

STYLE_MAP = {
    "вежливое": ["вежлив", "профессиональн", "официальн", "деловой"],
    "краткое": ["кратк", "лаконичн", "сжат"],
    "настойчивое": ["настойчив", "требовательн", "убедительн"],
}

In [12]:
# categories = list(TEMPLATES.keys())
# categories
# # length_cat = len(categories)
# # length_cat
# # n = 5500
# # per_cat = n // length_cat
# # remainder = n % length_cat
# # per_cat, remainder
# # for i,category in enumerate(categories):
# #     count = per_cat + (1 if i < remainder else 0)
# #     print(count)
# TEMPLATES['запрос_информации']
# topic = random.choice(TOPICS)
# topic
# instruction = random.choice(INSTRUCTIONS)
# instr_lower = instruction.lower()
# print(inst_lower)
# for style,keywords in STYLE_MAP.items():
#     print(style , keywords)
#     if any(kw in instr_lower for kw in keywords):
#       print("final",style)

In [13]:
# keywords = ['вежлив', 'профессиональн', 'официальн', 'деловой']
# instr_lower = "составьте вежливый профессиональный ответ на входящее деловое письмо."

# matches = [kw for kw in keywords if kw in instr_lower]

# print(matches)

In [14]:
def pick_style(instruction:str) -> str:
  instr_lower = instruction.lower()
  for style,keywords in STYLE_MAP.items():
    if any(kw in instr_lower for kw in keywords):
      return style
  return random.choice(["вежливое", "краткое", "настойчивое"])

In [15]:
def generate_example(category:str) -> dict:
  tmpl = TEMPLATES[category]
  topic = random.choice(TOPICS)
  date = random.choice(DATES)
  sender_raw = random.choice(SENDERS)
  sender_name = sender_raw.split(",")[0].strip()
  responder = random.choice(RESPONDERS)
  org = random.choice(ORGS)
  instruction = random.choice(INSTRUCTIONS)
  style = pick_style(instruction)

  incoming_tmpl = random.choice(tmpl["incoming"])

  incoming = incoming_tmpl.format(topic=topic, date=date, sender=sender_raw)
  response_body = random.choice(RESPONSE_BODIES[category])
  response_tmpl = tmpl["responses"][style]
  output = response_tmpl.format(
    sender=sender_name,
    topic=topic,
    date=date,
    response_body=response_body,
    responder=responder,
    org=org,
)
  return {
      "instruction": instruction,
      "input": incoming,
      "output": output,
      "category": category,
      "style": style,
  }
  # print(topic)
  # print("*"*50)
  # print(date)
  # print("*"*50)
  # print(sender_raw)
  # print("*"*50)
  # print(sender_name)
  # print("*"*50)
  # print(responder)
  # print("*"*50)
  # print(org)
  # print("*"*50)
  # print(instruction)
  # print("*"*50)
  # print(style)
  # print("*"*50)
  # print(incoming_tmpl)
  # print("*"*50)
  # print(incoming)
  # print("*"*50)
  # print(response_body)
  # print("*"*50)
  # print(response_tmpl)
  # print("*"*50)
  # print(output)

In [16]:
generate_example('запрос_информации')

{'instruction': 'Напишите краткий деловой ответ на следующее письмо.',
 'input': 'Уважаемые коллеги,\nПрошу предоставить информацию о обновления регламента работы с клиентами в срок до 15.02.2025.\nС уважением,\nА.В. Петрова, ООО «ТехноСервис»',
 'output': 'Уважаемый(-ая) А.В. Петрова,\n\nБлагодарим за обращение. В ответ на Ваш запрос относительно обновления регламента работы с клиентами сообщаем следующее:\n\nЗапрошенные данные по указанному вопросу будут предоставлены в течение 5 рабочих дней. Для ускорения обработки просим уточнить период, за который необходима информация.\n\nПри возникновении дополнительных вопросов просим обращаться к нам в рабочее время.\n\nС уважением,\nЛ.И. Захарова\nАО «ПартнёрИнвест»',
 'category': 'запрос_информации',
 'style': 'вежливое'}

In [17]:
def generate_dataset(n:int = 5500) -> List[dict]:
  categories = list(TEMPLATES.keys())
  dataset = []

  per_cat = n // len(categories)
  remainder = n % len(categories)

  for i,category in enumerate(categories):
    count = per_cat + (1 if i < remainder else 0)
    for _ in range(count):
      dataset.append(generate_example(category))

  random.shuffle(dataset)
  return dataset


In [18]:
dataset = generate_dataset(n=5500)
train = dataset[:5000]
val = dataset[5000:5250]
test = dataset[5250:5500]
print(len(val))

250


In [19]:
import os

os.makedirs("data",exist_ok=True)

with open("data/train.json","w",encoding="utf-8") as f:
  json.dump(train,f,ensure_ascii=False,indent=2)

with open("data/val.json","w",encoding="utf-8") as f:
  json.dump(val,f,ensure_ascii=False,indent=2)

with open("data/test.json","w",encoding="utf-8") as f:
  json.dump(test,f,ensure_ascii=False,indent=2)


In [20]:
print(f"✅ Датасет сгенерирован:")
print(f"   Train: {len(train)} примеров")
print(f"   Val:   {len(val)} примеров")
print(f"   Test:  {len(test)} примеров")
print(f"\nКатегории: {', '.join(TEMPLATES.keys())}")
print(f"\nПример из train[0]:")
ex = train[0]
print(f"  Instruction: {ex['instruction']}")
print(f"  Category:    {ex['category']} / {ex['style']}")
print(f"  Input[:80]:  {ex['input'][:80]}...")
print(f"  Output[:80]: {ex['output'][:80]}...")

✅ Датасет сгенерирован:
   Train: 5000 примеров
   Val:   250 примеров
   Test:  250 примеров

Категории: запрос_информации, жалоба, коммерческое_предложение, уведомление, согласование

Пример из train[0]:
  Instruction: Составьте вежливый профессиональный ответ на входящее деловое письмо.
  Category:    уведомление / вежливое
  Input[:80]:  Уважаемые коллеги,
Уведомляем Вас об изменениях в реструктуризации задолженности...
  Output[:80]: Уважаемый(-ая) Д.Р. Фёдоров,

Подтверждаем получение уведомления об изменениях в...


In [21]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [22]:
import argparse
import json
import time
import os
from dataclasses import dataclass

import torch
from datasets import Dataset
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    set_seed,
)

/Users/darkhanomirbay/Downloads/variant12/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0510 20:09:56.895000 4566 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [23]:
@dataclass
class Config:
  model_name: str
  mode: str

  lora_r:int = 16
  lora_alpha:int = 32
  lora_dropout:float = 0.05
  target_modules:tuple = ("q_proj","v_proj","k_proj","o_proj")

  max_seq_length: int = 512
  per_device_train_batch_size:int = 4
  per_device_eval_batch_size:int = 4
  gradient_accumulation_steps:int = 4
  num_train_epochs:int = 3
  learning_rate:float = 2e-4
  warmup_ratio:float = 0.03
  lr_scheduler_type:str = "cosine"
  fp16:bool = True
  bf16:bool = False

  train_path: str = "data/train.json"
  val_path: str = "data/val.json"
  output_dir: str = "results/lora_adapter"
  logging_dir: str = "results/logs"

  seed: int = 42


In [24]:
# cfg = Config(
#         model_name="IlyaGusev/saiga_mistral_7b",
#         mode="qlora",
#         num_train_epochs=3,
#         lora_r=8,
#         lora_alpha=16,
#         fp16=True,
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=8,
#         train_path="/content/train.json",
#         val_path="/content/val.json",
#         output_dir="/content/drive/MyDrive/lora_adapter",
#         target_modules=("q_proj", "v_proj"),
#         max_seq_length=256

# )

cfg = Config(
    model_name="ai-forever/rugpt3large_based_on_gpt2",
    mode="lora",
    target_modules=("c_attn",),
    lora_r=8,
    lora_alpha=16,
    fp16=True,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_seq_length=256,
    num_train_epochs=3,
    train_path="./data/train.json",
    val_path="./data/val.json",
    output_dir="./lora_adapter",
)

In [25]:
cfg

Config(model_name='ai-forever/rugpt3large_based_on_gpt2', mode='lora', lora_r=8, lora_alpha=16, lora_dropout=0.05, target_modules=('c_attn',), max_seq_length=256, per_device_train_batch_size=4, per_device_eval_batch_size=4, gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=0.0002, warmup_ratio=0.03, lr_scheduler_type='cosine', fp16=True, bf16=False, train_path='./data/train.json', val_path='./data/val.json', output_dir='./lora_adapter', logging_dir='results/logs', seed=42)

In [26]:
def load_model_and_tokenizer(cfg:Config):
  tokenizer = AutoTokenizer.from_pretrained(cfg.model_name,use_fast=False,padding_side="right")

  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  if cfg.mode == "qlora":
    print("Mode: QLoRA (4-bit NF4)")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_compute_dtype = torch.float16,
        bnb_4bit_use_double_quant = True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        quantization_config = bnb_config,
        device_map = "auto",
        trust_remote_code = True,
    )
    model = prepare_model_for_kbit_training(model)
  else:
    print("🔧 Режим: LoRA (FP16)")
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

  return model,tokenizer


In [27]:
def apply_lora(model,cfg:Config):
  lora_config = LoraConfig(
      task_type=TaskType.CAUSAL_LM,
      r=cfg.lora_r,
      lora_alpha=cfg.lora_alpha,
      lora_dropout=cfg.lora_dropout,
      target_modules=cfg.target_modules)
  model = get_peft_model(model,lora_config)
  model.print_trainable_parameters()
  return model


In [28]:
def load_data(path:str) -> Dataset:
  with open(path,"r",encoding="utf-8") as f:
    data = json.load(f)
  return Dataset.from_list(data)

In [29]:
PROMPT_TEMPLATE_INFERENCE = (
    "### Инструкция:\n{instruction}\n\n"
    "### Входящее письмо:\n{input}\n\n"
    "### Ответ:\n"
)

In [30]:
PROMPT_TEMPLATE = (
    "### Инструкция:\n{instruction}\n\n"
    "### Входящее письмо:\n{input}\n\n"
    "### Ответ:\n{output}"
)

In [31]:
def format_prompt(example:dict,for_inference:bool = False) -> str:
  if for_inference:
    return PROMPT_TEMPLATE_INFERENCE.format(
            instruction=example["instruction"],
            input=example["input"],
    )
  return PROMPT_TEMPLATE.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"],
    )

In [32]:
def tokenize_dataset(dataset:Dataset,tokenizer,max_length:int) -> Dataset:
  def tokenize(example):
    full_text = format_prompt(example,for_inference=False)

    prefix = format_prompt(example, for_inference=True)

    full = tokenizer(
            full_text,
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    prefix_len = len(tokenizer(prefix, truncation=True, max_length=max_length)["input_ids"])

    labels = full["input_ids"].copy()
    labels[:prefix_len] = [-100] * prefix_len

    full["labels"] = labels
    return full
  return dataset.map(tokenize, remove_columns=dataset.column_names)


In [33]:
def train(cfg:Config):
  set_seed(cfg.seed)

  print(f"\n{'='*60}")
  print(f"  Модель:   {cfg.model_name}")
  print(f"  Режим:    {cfg.mode.upper()}")
  print(f"  Train:    {cfg.train_path}")
  print(f"  Output:   {cfg.output_dir}")
  print(f"{'='*60}\n")

  model,tokenizer = load_model_and_tokenizer(cfg)

  model = apply_lora(model,cfg)

  train_ds = tokenize_dataset(load_data(cfg.train_path), tokenizer, cfg.max_seq_length)
  val_ds = tokenize_dataset(load_data(cfg.val_path), tokenizer, cfg.max_seq_length)
  
  print(f"Train tokens: {sum(len(x) for x in train_ds['input_ids']):,}")
  print(f"Val   tokens: {sum(len(x) for x in val_ds['input_ids']):,}")

  # DataCollator
  data_collator = DataCollatorForSeq2Seq(
      tokenizer=tokenizer,
      model=model,
      padding=True,
      pad_to_multiple_of=8,
  )

  # Аргументы обучения
  training_args = TrainingArguments(
      output_dir=cfg.output_dir,
      per_device_train_batch_size=cfg.per_device_train_batch_size,
      per_device_eval_batch_size=cfg.per_device_eval_batch_size,
      gradient_accumulation_steps=cfg.gradient_accumulation_steps,
      num_train_epochs=cfg.num_train_epochs,
      learning_rate=cfg.learning_rate,
      warmup_ratio=cfg.warmup_ratio,
      lr_scheduler_type=cfg.lr_scheduler_type,
      fp16=cfg.fp16,
      bf16=cfg.bf16,
      logging_dir=cfg.logging_dir,
      logging_steps=50,
      eval_strategy="epoch",
      save_strategy="epoch",
      load_best_model_at_end=True,
      metric_for_best_model="eval_loss",
      report_to="none",  # поменять на "wandb" при наличии
      dataloader_num_workers=2,
      seed=cfg.seed,
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      processing_class=tokenizer,
      data_collator=data_collator,
  )

  # Обучение
  start = time.time()
  trainer.train()
  elapsed = time.time() - start
  print(f"\n✅ Обучение завершено за {elapsed/60:.1f} мин.")

  # Сохранение адаптера
  os.makedirs(cfg.output_dir, exist_ok=True)
  trainer.save_model(cfg.output_dir)
  tokenizer.save_pretrained(cfg.output_dir)
  print(f"💾 Адаптер сохранён в: {cfg.output_dir}")

  return trainer

In [34]:
train(cfg)



  Модель:   ai-forever/rugpt3large_based_on_gpt2
  Режим:    LORA
  Train:    ./data/train.json
  Output:   ./lora_adapter

🔧 Режим: LoRA (FP16)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 845.87it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/darkhanomirbay/Downloads/variant12/.venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 1,179,648 || all params: 761,479,680 || trainable%: 0.1549


Map: 100%|██████████| 250/250 [00:00<00:00, 1175.58 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Train tokens: 843,784
Val   tokens: 41,668


/Users/darkhanomirbay/Downloads/variant12/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.123157,0.086219
2,0.085761,0.070729
3,0.080148,0.069025


/Users/darkhanomirbay/Downloads/variant12/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/darkhanomirbay/Downloads/variant12/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



✅ Обучение завершено за 71.7 мин.
💾 Адаптер сохранён в: ./lora_adapter


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")
tokenizer = AutoTokenizer.from_pretrained("./lora_adapter")
model = PeftModel.from_pretrained(base, "./lora_adapter")
model.eval()

/Users/darkhanomirbay/Downloads/variant12/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0510 22:17:33.845000 8680 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 57437.42it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1536)
        (wpe): Embedding(2048, 1536)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-23): 24 x GPT2Block(
            (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=4608, nx=1536)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4608, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
     

In [3]:
def generate_reply(incoming_letter, instruction="Составьте вежливый профессиональный ответ на входящее деловое письмо."):
    prompt = f"""### Инструкция:
{instruction}

### Входящее письмо:
{incoming_letter}

### Ответ:
"""
    inputs = tokenizer(prompt, return_tensors="pt")
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = generated[len(prompt):]
    
    # Обрезаем по первому стоп-паттерну
    stop_patterns = [
        "### ", "==", "\nA:", "\nQ:",
        "ПАО «", "ООО «", "Агентство", "Партнёрство",
        "г. Санкт", "г. Москва",
        "Финансовый блок",          # ← новое
        "Нашими основными",         # ← новое  
        "Тел.:", "Факс:",           # ← новое
        "…\n",                      # ← новое
    ]
    for stop in stop_patterns:
        if stop in answer:
            answer = answer[:answer.index(stop)]
    
    return answer.strip()

In [43]:
tests = [
    ("запрос", "Уважаемые коллеги,\nПрошу предоставить финансовый отчёт за Q3 до 15.02.2025.\nС уважением, А.В. Петрова"),
    ("жалоба", "Добрый день,\nВыражаем обеспокоенность в связи с нарушением сроков поставки по договору №123.\nС уважением, И.И. Иванов"),
    ("предложение", "Уважаемые партнёры,\nПредлагаем рассмотреть сотрудничество в сфере IT-аутсорсинга.\nС уважением, М.С. Сидорова"),
]

for name, letter in tests:
    print(f"\n{'='*50}")
    print(f"📨 Тип: {name}")
    print(f"Входящее: {letter}...")
    print(f"\n📝 Ответ модели:")
    print(generate_reply(letter))


📨 Тип: запрос
Входящее: Уважаемые коллеги,
Прошу предоставить финансовый отчёт за Q3 до 15.02.2025.
С уважением, А.В. Петрова...

📝 Ответ модели:
Уважаемый(-ая) А.В. Петрова,

Благодарим Вас за обращение. В ответ на Ваш запрос относительно финансовой отчетности сообщаем следующее:

Запрошенные данные по указанному вопросу будут предоставлены в течение 5 рабочих дней после внутреннего согласования с профильным отделом. Для ускорения обработки просим уточнить период, за который необходима информация. Если он не будет совпадать со временем текущей задачи — это вызовет дополнительные неудобства для сотрудников финансового блока и повлечёт дополнительное времяпрепровождение внутри группы. Просим дополнительно подтвердить получение уведомления о возникших проблемах при помощи внутренней формы.

При возникновении дополнительных вопросов просим обращаться к нам в рабочее время. С уважением, Ф.О. Лебедев

📨 Тип: жалоба
Входящее: Добрый день,
Выражаем обеспокоенность в связи с нарушением сроков

In [4]:
import json, random, re
import pandas as pd

random.seed(99)

# ── Rubric helpers ────────────────────────────────────────────────
FORMAL_MARKERS = [
    "уважаемый", "уважаемая", "с уважением", "благодарим",
    "сообщаем", "настоящим", "просим", "направляем",
]
HALLUCINATION_RISK = [
    "гарантируем возврат", "штраф составит", "уголовная ответственность",
    "лично явитесь", "немедленно оплатите",
]

def score_style(reply: str) -> int:
    r = reply.lower()
    hits = sum(1 for m in FORMAL_MARKERS if m in r)
    if hits >= 4:
        return 5
    elif hits >= 2:
        return 4
    elif hits >= 1:
        return 3
    else:
        return 2

def score_relevance(reply: str, topic: str) -> int:
    topic_words = [w for w in re.split(r"\W+", topic.lower()) if len(w) > 4]
    r = reply.lower()
    hits = sum(1 for w in topic_words if w in r)
    if hits >= 2:
        return 5
    elif hits == 1:
        return 4
    elif len(reply) > 100:
        return 3
    else:
        return 2

def score_hallucination(reply: str) -> int:
    r = reply.lower()
    if any(h in r for h in HALLUCINATION_RISK):
        return 2
    if len(reply) > 400:   # overgeneration risk
        return 3
    return 5

# ── Load test set and sample 30 ───────────────────────────────────
with open("data/test.json", encoding="utf-8") as f:
    test_data = json.load(f)

# Ensure coverage: pick at least 2 per (category × style), then random fill
by_key = {}
for ex in test_data:
    k = (ex["category"], ex["style"])
    by_key.setdefault(k, []).append(ex)

sample = []
for examples in by_key.values():
    sample.extend(random.sample(examples, min(2, len(examples))))

remaining = [ex for ex in test_data if ex not in sample]
if len(sample) < 30:
    sample.extend(random.sample(remaining, 30 - len(sample)))
sample = sample[:30]

# ── Run inference + score ─────────────────────────────────────────
rows = []
for i, ex in enumerate(sample, 1):
    reply = generate_reply(ex["input"], ex["instruction"])

    # extract topic hint from input (first noun phrase-ish token)
    topic_hint = ex.get("category", "")

    s_style  = score_style(reply)
    s_rel    = score_relevance(reply, ex["input"])
    s_hall   = score_hallucination(reply)
    avg      = round((s_style + s_rel + s_hall) / 3, 2)

    rows.append({
        "#":          i,
        "category":   ex["category"],
        "style":      ex["style"],
        "reply_preview": reply[:100].replace("\n", " ") + "…",
        "style_score":   s_style,
        "relevance":     s_rel,
        "no_halluc":     s_hall,
        "avg":           avg,
    })

df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)
print(df.to_string(index=False))

print("\n" + "=" * 60)
print("  ИТОГОВЫЕ ПОКАЗАТЕЛИ (30 тестовых примеров)")
print("=" * 60)
print(f"  Стиль:                    {df['style_score'].mean():.2f} / 5")
print(f"  Релевантность:            {df['relevance'].mean():.2f} / 5")
print(f"  Отсутствие галлюцинаций:  {df['no_halluc'].mean():.2f} / 5")
print(f"  ОБЩАЯ СРЕДНЯЯ:            {df['avg'].mean():.2f} / 5")
print("=" * 60)

print("\nПо категориям:")
print(df.groupby("category")["avg"].mean().round(2).to_string())

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 #                 category       style                                                                                         reply_preview  style_score  relevance  no_halluc  avg
 1              уведомление    вежливое Уважаемый(-ая) Д.Р. Фёдоров,  Изменения по устраннения технической неполадки оборудования приняты к …            4          5          3 4.00
 2              уведомление    вежливое Уважаемый(-ая) О.Н. Новикова,  Подтверждаем получение уведомления о новых изменениях в согласования …            4          5          3 4.00
 3 коммерческое_предложение    вежливое Уважаемый(-ая) К.В. Морозов,  Благодарим за проявленный интерес к сотрудничеству. Ваше участие — бес…            5          5          3 4.33
 4 коммерческое_предложение    вежливое Уважаемый(-ая) О.Н. Новикова,  Благодарим за проявленный интерес к сотрудничеству. Ваше предложение …            4          5          3 4.00
 5        запрос_информации    вежливое Уважаемый(-ая) А.В. Петрова,  Благодарим за обраще

## Экспертная оценка (30 тестовых примеров)

Шкала 1–5 по трём осям: **стиль**, **релевантность**, **отсутствие галлюцинаций**.

In [5]:
import time

BENCHMARK_LETTER = (
    "Уважаемые коллеги,\n"
    "Прошу предоставить финансовый отчёт за Q3 2024 в срок до 15.02.2025.\n"
    "С уважением, А.В. Петрова, ООО «ТехноСервис»"
)
BENCHMARK_INSTRUCTION = "Составьте вежливый профессиональный ответ на входящее деловое письмо."
N_RUNS = 10

latencies = []
token_counts = []

prompt = f"### Инструкция:\n{BENCHMARK_INSTRUCTION}\n\n### Входящее письмо:\n{BENCHMARK_LETTER}\n\n### Ответ:\n"
inputs = tokenizer(prompt, return_tensors="pt")

for i in range(N_RUNS):
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,          # greedy для воспроизводимости
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    t1 = time.time()
    n_new = out.shape[1] - inputs["input_ids"].shape[1]
    latencies.append(t1 - t0)
    token_counts.append(n_new)

avg_lat = sum(latencies) / len(latencies)
med_lat = sorted(latencies)[len(latencies) // 2]
p95_lat = sorted(latencies)[int(len(latencies) * 0.95)]
avg_toks = sum(token_counts) / len(token_counts)
avg_tps = avg_toks / avg_lat

print("=" * 50)
print("  ЗАМЕР СКОРОСТИ ИНФЕРЕНСА")
print("=" * 50)
print(f"  Прогонов:             {N_RUNS}")
print(f"  Avg tokens/reply:     {avg_toks:.1f}")
print(f"  Latency mean:         {avg_lat:.2f} сек")
print(f"  Latency median:       {med_lat:.2f} сек")
print(f"  Latency p95:          {p95_lat:.2f} сек")
print(f"  Tokens/sec (mean):    {avg_tps:.1f} tok/s")
print("=" * 50)

  ЗАМЕР СКОРОСТИ ИНФЕРЕНСА
  Прогонов:             10
  Avg tokens/reply:     200.0
  Latency mean:         22.31 сек
  Latency median:       22.23 сек
  Latency p95:          23.51 сек
  Tokens/sec (mean):    9.0 tok/s


## Замер скорости инференса